In [ ]:
import sys
sys.path.append('../')

In [ ]:
%load_ext autoreload
%autoreload 2

In [47]:
import numpy as np
import pandas as pd
import torch
import re

from typing import Callable, Iterable, List, Tuple
from tokenizers import ByteLevelBPETokenizer

from src.config.paths import DATA_PATH
from src.config.lex import N_VOCAB, CHUNK_SIZE

# Dataset

In [32]:
df = pd.read_parquet(DATA_PATH.joinpath('dia-final.parquet'))

In [33]:
df

,chunk_id,text,pos_id,neg_id
id,,,,
1681_0.q_0,0,I think this is your biggest success right now...,1681_1.q_1,1681_8.a_1
1681_0.a_0,0,Yeah.,1681_1.a_1,1681_9.q_0
1681_1.q_0,0,How would you describe it?,1681_2.q_0,1681_0.a_0
1681_1.q_1,0,Is it fantastic for you?,1681_2.q_0,1681_0.a_0
1681_1.a_0,0,"Yeah, I'm pretty happy, but it was --",1681_2.a_1,1681_0.q_0
...,...,...,...,...
24959__4_000_21,16218,"Ah, the participants, the beneficiaries, they'...",24959__4_000_20,24959__3_171_4
24959__4_000_22,16218,The plan is the beneficiary.,24959__4_000_21,24959__3_171_4
24959__4_000_23,16218,"If he's right, we lose.",24959__4_000_22,24959__3_171_4


## Class Implementation

In [ ]:
class LexDataset(torch.utils.data.Dataset):
    def __init__(self, df:pd.DataFrame, tokenize_fn, mutate_fn=None):
        self.tokenize_fn = tokenize_fn
        self.mutate_fn = mutate_fn
        self.df = df
        self.n_chunks = self.df['chunk_id'].nunique()
    
    def __getitem__(self, idx) -> Iterable[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]]:
        '''Get In-sequence Conversation Chunk alongside Positives and Negatives for each Chunk Segment'''
        chunk = self.df[self.df['chunk_idx'] == idx]

        texts = chunk['text']
        pos = chunk['pos_id']
        neg = chunk['neg_id']
        
        stoi = { id : i for i, id in enumerate(self.df.index) }

        texts = [
            torch.tensor(self.tokenize_fn(
                self.mutate_fn(sent) if self.mutate_fn else sent
            ))
            for sent in texts
        ]
        pos = [stoi[i] for i in pos]
        neg = [stoi[i] for i in neg]

        return list(zip(texts, pos, neg))

    def __len__(self) -> int:
        return self.n_chunks

## Tokenizers

In [ ]:
# with open(DATA_PATH.joinpath('flattened_corpora.txt'), 'w+') as f:
#     for i in [IQ2_PATH, SUPREME_PATH, TENNIS_PATH]:
#         df = pd.read_json(i.joinpath('utterances.jsonl'), lines=True)
#         f.write('\n'.join(df['text']))

In [ ]:
tok = ByteLevelBPETokenizer()

In [ ]:
tok.train(
    files=[DATA_PATH.joinpath('flattened_corpora.txt').__str__()],
    vocab_size=N_VOCAB,
    min_frequency=2,
    special_tokens=['[SEP]']
)

## Mutator

In [ ]:
def mutate(text:str) -> str:
    rand = torch.rand(1)[0]
    if rand > 0.20:
        return text
    if rand > 0.10:
        return text.lower()
    
    if rand > 0.05:
        text = text.lower()
    return ''.join(re.split(r'[.,;:!?]+', text))

# Model